In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
# Load Parquet file
rule_of_law_df = pd.read_parquet('../data/image_data/wb_ruleoflaw.parquet')
share_of_industry_df = pd.read_parquet('../data/image_data/Share_of_industry.parquet')
tax_revenue_df = pd.read_parquet('../data/image_data/Tax_revenue.parquet')
inflation_df = pd.read_parquet('../data/image_data/Inflation.parquet')
edu_spending_df = pd.read_parquet('../data/image_data/Education_spending.parquet')
govermment_effective_df = pd.read_parquet('../data/image_data/wb_government_effectiveness.parquet')

In [5]:
rule_of_law_df = rule_of_law_df.reset_index().rename(columns={"index": "Year"})
share_of_industry_df = share_of_industry_df.reset_index().rename(columns={"index": "Year"})
tax_revenue_df = tax_revenue_df.reset_index().rename(columns={"index": "Year"})
inflation_df = inflation_df.reset_index().rename(columns={"index": "Year"})
edu_spending_df = edu_spending_df.reset_index().rename(columns={"index": "Year"})
govermment_effective_df = govermment_effective_df.reset_index().rename(columns={"index": "Year"})

In [6]:
name_df = [
    'rule_of_law_df', 'share_of_industry_df', 'tax_revenue_df', 'inflation_df', 'edu_spending_df', 'govermment_effective_df'
]

In [11]:
list_df = [
    rule_of_law_df, share_of_industry_df, tax_revenue_df, inflation_df, edu_spending_df, govermment_effective_df
]

In [9]:
def remove_columns(df, columns_to_remove):
    """
    Removes specific columns from a DataFrame.

    Parameters:
    - df: Original DataFrame
    - columns_to_remove: list of column names to remove

    Returns:
    - DataFrame without the specified columns
    """
    return df.drop(columns=columns_to_remove, errors='ignore')

In [10]:
# countries to delete
delete_countries = ['China', 'Morocco', 'Macedonia', 'Indonesia']

rule_of_law_df = remove_columns(rule_of_law_df, delete_countries)
share_of_industry_df = remove_columns(share_of_industry_df, delete_countries)
tax_revenue_df = remove_columns(tax_revenue_df, delete_countries)
inflation_df = remove_columns(inflation_df, delete_countries)
edu_spending_df = remove_columns(edu_spending_df, delete_countries)
govermment_effective_df = remove_columns(govermment_effective_df, delete_countries)

In [12]:
def check_columns_in_filenames(dataframes, df_names, path):
    """
    Checks whether column names from each DataFrame exist in the filenames of a given path,
    accounting for differences like '-' in column names vs '_' in filenames.

    Parameters:
    - dataframes: list of pandas DataFrames
    - df_names: list of names corresponding to each DataFrame
    - path: directory path containing files

    Prints:
    - For each DataFrame, its name followed by the list of column names
      that do NOT appear in any of the filenames in the specified path.
    """
    try:
        filenames = os.listdir(path)
    except FileNotFoundError:
        print(f"❌ Path not found: {path}")
        return

    # Normalize filenames: lowercase and replace underscores with hyphens
    normalized_filenames = [f.lower().replace("_", "-") for f in filenames]

    for df, name in zip(dataframes, df_names):
        missing_columns = []
        for col in df.columns:
            normalized_col = col.lower().replace("_", "-")
            if not any(normalized_col in fname for fname in normalized_filenames):
                missing_columns.append(col)

        print(f"📊 DataFrame: {name}")
        if missing_columns:
            print(f"   ❌ Columns not found in filenames: {missing_columns}")
        else:
            print("   ✅ All columns found in filenames")

In [14]:
check_columns_in_filenames(list_df, name_df, '../data/by_country')

📊 DataFrame: rule_of_law_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: share_of_industry_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: tax_revenue_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: inflation_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: edu_spending_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: govermment_effective_df
   ❌ Columns not found in filenames: ['Year']


In [15]:
def update_country_parquets_by_name_match(macro_dict, output_path):
    os.makedirs(output_path, exist_ok=True)
    existing_files = [f for f in os.listdir(output_path) if f.endswith('.parquet')]

    for indicator, df in macro_dict.items():
        for country in df.columns:
            if country == 'Year':
                continue

            # Solo procesar si el nombre del país contiene "-"
            if "-" not in country:
                continue

            # Normalizar nombre del país para emparejar con archivos
            normalized_country = country.replace("-", "_")

            # Buscar archivo que contenga el nombre normalizado
            matched_file = next(
                (f for f in existing_files if normalized_country in f),
                None
            )

            if matched_file:
                file_path = os.path.join(output_path, matched_file)
                existing_df = pd.read_parquet(file_path)

                # Extraer columna del país y renombrar con el indicador
                country_data = df[['Year', country]].copy()
                country_data.rename(columns={country: indicator}, inplace=True)

                # Actualizar archivo existente
                updated_df = pd.merge(existing_df, country_data, on='Year', how='outer')
                updated_df.sort_values(by='Year', inplace=True)
                updated_df.to_parquet(file_path, index=False)
            else:
                print(f"⚠️ No match found for country: {country}")

In [16]:
macro_dict = {
    'rule_of_law_df': rule_of_law_df,
    'share_of_industry_df': share_of_industry_df,
    'tax_revenue_df': tax_revenue_df,
    'inflation_df': inflation_df,
    'edu_spending_df': edu_spending_df,
    'govermment_effective_df': govermment_effective_df
}

In [17]:
update_country_parquets_by_name_match(macro_dict, '../data/by_country')

In [18]:
czech_df_01 = pd.read_parquet('../data/by_country/Czech_Republic_macro_profile.parquet')
hongkong_df_01 = pd.read_parquet('../data/by_country/Hong_Kong_macro_profile.parquet')
new_zealand_df_01 = pd.read_parquet('../data/by_country/New_Zealand_macro_profile.parquet')
south_a_df_01 = pd.read_parquet('../data/by_country/South_Africa_macro_profile.parquet')
south_k_df_01 = pd.read_parquet('../data/by_country/South_Korea_macro_profile.parquet')
uk_df_01 = pd.read_parquet('../data/by_country/United_Kingdom_macro_profile.parquet')

In [19]:
czech_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,286.0300,NaN,964.1600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,291.7725,NaN,1009.3900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,268.1400,NaN,986.3775,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,267.4050,NaN,1008.2400,NaN,NaN,99.9975,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,267.2050,NaN,1023.8400,NaN,NaN,100.0250,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,261.2600,97.975000,1029.5975,NaN,NaN,98.5550,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,920.626667,258.2225,99.341667,1042.2525,NaN,NaN,98.5800,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,1182.200000,270.7900,99.750000,1094.4975,2072.538333,NaN,100.9950,NaN,NaN,NaN,NaN,3.67,NaN
8,2015,NaN,101.3,45.25,NaN,220.0425,1431.200833,298.6050,100.008333,1162.9525,2202.911667,NaN,105.0125,NaN,NaN,14.77,NaN,5.75,NaN
9,2016,5.35,101.8,45.18,73.57,228.7500,1902.431667,294.3300,100.658333,1210.7600,2356.845833,3.966667,112.5225,0.99,NaN,14.91,NaN,5.55,1.01


In [20]:
hongkong_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,81.3450,NaN,403.8975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,83.5875,NaN,419.2550,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,80.6825,NaN,405.5500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,93.6425,NaN,437.0250,NaN,NaN,100.0000,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,113.8250,NaN,483.6100,NaN,NaN,120.6350,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,129.3525,89.625000,509.2650,NaN,NaN,136.6450,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,305.482500,128.8800,93.500000,534.5775,NaN,NaN,160.6000,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,322.330833,132.7300,97.650000,565.0000,4376.982500,NaN,170.2075,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,NaN,96.2,NaN,NaN,57.8175,342.510000,134.3025,100.575000,599.5700,4688.043333,NaN,196.6575,NaN,NaN,NaN,NaN,NaN,NaN
9,2016,3.96,107.5,NaN,100.0,61.9925,366.943333,133.8050,103.000000,622.6100,4952.230000,3.383333,189.5900,1.68,7.48,NaN,NaN,3.29,1.80


In [21]:
new_zealand_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,11.0325,NaN,45.8300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,10.9850,NaN,45.7650,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,9.8975,NaN,46.4225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,8.2050,NaN,48.7800,NaN,NaN,99.9975,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,9.2850,NaN,52.8100,NaN,NaN,100.7500,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,10.0200,952.691675,53.9900,NaN,NaN,105.2500,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,21.706667,11.3175,963.499175,56.9175,NaN,NaN,114.5900,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,21.696667,13.5550,975.326250,59.8825,536.010833,NaN,122.0325,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,NaN,101.7,40.68,NaN,10.4975,24.615000,14.0075,978.181075,61.8875,536.010833,NaN,137.1775,NaN,NaN,27.39,NaN,5.71,NaN
9,2016,2.65,99.7,40.31,86.40,11.7650,25.075000,15.5125,984.502450,66.1525,522.752500,4.425000,156.6250,1.91,20.17,27.25,NaN,5.51,1.81


In [22]:
south_a_df_01 

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,111.5925,NaN,571.8375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,141.1250,NaN,652.4600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,136.1525,NaN,691.1775,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,134.1550,NaN,757.1800,NaN,NaN,100.0000,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,148.1325,NaN,837.9275,NaN,NaN,103.5900,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,159.9100,78.425000,895.1825,NaN,NaN,108.4075,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,480.495000,179.6850,82.925000,967.1575,NaN,NaN,115.4075,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,535.531667,189.1350,87.975000,1033.4675,2564.023333,NaN,123.9875,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,NaN,101.5,79.42,NaN,209.8225,599.556667,199.0350,92.000000,1105.2025,2769.898333,NaN,131.7975,NaN,NaN,24.29,NaN,NaN,NaN
9,2016,23.79,100.2,79.42,65.34,229.7375,678.905000,207.5375,97.833333,1189.8875,2958.854167,27.450,138.2250,-0.16,NaN,24.06,NaN,5.44,0.10


In [23]:
south_k_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,82142.900,NaN,272415.050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,89439.875,NaN,288554.150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,93013.925,NaN,301336.950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,99946.475,NaN,330652.800,NaN,NaN,99.9975,NaN,NaN,NaN,NaN,3.53,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,104706.175,NaN,347234.300,NaN,NaN,105.2275,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,106403.450,91.815083,360027.825,NaN,NaN,108.2500,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,333.290000,109157.000,93.009917,375204.800,NaN,NaN,107.8000,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,360.684167,113147.525,94.195583,390732.250,1.656976e+06,NaN,109.4325,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,NaN,101.6,17.82,NaN,62522.000,368.030000,120250.425,94.860917,414505.100,1.772220e+06,NaN,113.0825,NaN,NaN,13.16,NaN,4.45,NaN
9,2016,28.05,100.2,17.45,81.56,66323.825,371.585833,129337.475,95.782667,435194.900,1.905550e+06,3.708333,116.0900,1.13,34.30,13.98,NaN,4.33,1.02


In [24]:
uk_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,70.0750,NaN,386.1575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,70.3125,NaN,398.4000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,62.5150,NaN,387.2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,64.5750,NaN,402.1400,NaN,NaN,100.0000,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,65.3075,NaN,415.6475,NaN,NaN,98.5450,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,67.5025,96.041667,428.4300,NaN,NaN,98.9200,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,129.022500,71.1100,98.508333,445.3400,NaN,NaN,101.4550,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,134.969167,77.1675,99.958333,465.6300,1594.228333,NaN,109.6025,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,NaN,100.5,70.84,NaN,93.6600,149.983333,83.2450,100.008333,479.1125,1618.274167,NaN,116.1250,NaN,NaN,25.05,NaN,5.55,NaN
9,2016,33.88,100.9,71.71,82.89,95.6900,166.655833,89.1525,100.650000,497.9125,1684.606667,4.891667,124.2700,1.64,NaN,25.44,NaN,5.43,1.57
